### Setting

In [ ]:
import itertools
import pickle
import sys
from pathlib import Path

import cpnet
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from tqdm.auto import tqdm

# --- repository paths -------------------------------------------------------
# `pip install -e .` from the repository root makes this import work anywhere;
# the fallback covers a plain `jupyter lab` started without installing.
try:
    from scisoc.config import paths
except ModuleNotFoundError:
    _here = Path.cwd().resolve()
    _root = next(p for p in (_here, *_here.parents) if (p / "src" / "scisoc").is_dir())
    sys.path.insert(0, str(_root / "src"))
    from scisoc.config import paths

paths.ensure()
print(paths.describe())

# The raw subject tables live outside the repository (external drive).
# Point at them with SCISOC_RAW in the environment or in <repo>/.env, e.g.
#     SCISOC_RAW=/Volumes/My Passport_2/Science-Society/data/processed
NEWS_PATH    = paths.raw_subjects("news")     # news_subject_by_year.pkl
PAPER_PATH   = paths.raw_subjects("paper")    # paper_subject_by_year.pkl

NETWORKS_DIR = paths.networks                 # networks/<source>/adj_<year>.npz + node_info.pkl
BACKBONE_DIR = paths.backbone                 # backbone/<source>_<year>_backbone.parquet
EMB_DIR      = paths.embeddings               # <source>_E.npz from src/dysat/train.py
CP_DIR       = paths.cp_results
PLOT_DIR     = paths.figures


### Data import

In [ ]:
from scisoc.io import load_node_info

info = {"news": load_node_info("news", NETWORKS_DIR),
        "paper": load_node_info("paper", NETWORKS_DIR)}

# sanity check
print(len(list(NETWORKS_DIR.glob("*/adj_*.npz"))), "matrices")
print(info["news"].keys())
print("news  years", len(info["news"]["years"]), "V_t", info["news"]["V_t"][0], "->", info["news"]["V_t"][-1])
print("paper years", len(info["paper"]["years"]), "V_t", info["paper"]["V_t"][0], "->", info["paper"]["V_t"][-1])

### Core-periphery

In [ ]:
def node_mask(info, min_share, max_share):
    """Concept filter on the full-period share."""
    s = info["concept_share"]
    return (s >= min_share) & (s <= max_share)

def cp_network(W, keep):
    """
    Apply the concept mask and drop isolates.
    Returns the submatrix and the global index of each of its rows.
    """
    idx = np.where(keep)[0]
    sub = W[idx][:, idx].tocsr()
    act = np.where(sub.getnnz(axis=1) > 0)[0]
    return sub[act][:, act].tocsr(), idx[act]

def detect_once(Wc, gidx, method="km_er"):
    """
    One independent label-switching run. default is num_runs=10
    """
    algo = (cpnet.KM_ER(num_runs=1) if method == "km_er"
            else cpnet.KM_config(num_runs=1))
    algo.detect(Wc)

    c = np.asarray(algo.c_)                      # pair id per node
    x = np.asarray(algo.x_).astype(bool)         # core flag per node

    core_local = np.where(x)[0]
    sizes = np.bincount(c[core_local])
    sizes = sizes[sizes > 0]                     # groups holding core nodes

    return {
        "core": gidx[core_local],                # global indices
        "S": int(len(sizes)),
        "H": float(((sizes / sizes.sum()) ** 2).sum()) if sizes.size else np.nan,
        "Q_cp": float(algo.Q_),
    }

def run_cell(W, keep, n_runs, method="km_er", min_nodes=10):
    """One (year, spec) cell: mask once, then run detection n_runs=10 times."""
    Wc, gidx = cp_network(W, keep)
    if Wc.shape[0] < min_nodes:
        return None
    return {
        "V_t": int(Wc.shape[0]),
        "E_t": int(Wc.nnz // 2),
        "runs": [detect_once(Wc, gidx, method) for _ in range(n_runs)],
    }

def run_grid(source, info, grid, net_dir, out_dir,
             n_runs=1, method="km_er", tag="grid"):
    """
    Year-outer, spec-inner so each yearly matrix is read once.
    Saves per cell so an interrupted sweep resumes where it stopped.
    """
    out_dir = Path(out_dir) / tag
    out_dir.mkdir(parents=True, exist_ok=True)
    masks = {spec: node_mask(info, *spec) for spec in grid}

    todo = [(y, s) for y in info["years"] for s in grid
            if not (out_dir / f"{source}_{y}_{s[0]}_{s[1]}.pkl").exists()]
    bar = tqdm(total=len(todo), desc=f"{source}/{tag}")

    for year in info["years"]:
        pending = {s: m for s, m in masks.items()
                   if not (out_dir / f"{source}_{year}_{s[0]}_{s[1]}.pkl").exists()}
        if not pending:
            continue
        W = load_npz(Path(net_dir) / source / f"adj_{year}.npz")
        for spec, keep in pending.items():
            cell = run_cell(W, keep, n_runs, method)
            with open(out_dir / f"{source}_{year}_{spec[0]}_{spec[1]}.pkl", "wb") as f:
                pickle.dump(cell, f)
            bar.set_postfix(year=int(year), spec=f"{spec[0]}-{spec[1]}",
                            V=cell["V_t"] if cell else 0)
            bar.update(1)
    bar.close()
                
def select_spec(df, agg="median"):
    """Pick the grid cell with the highest representative Q_cp, per arena."""
    g = df.groupby(["min_share", "max_share"])["Q_cp"]
    s = g.median() if agg == "median" else g.mean()
    return s.idxmax(), s.sort_values(ascending=False)

In [ ]:
def compute_indicators(source, info, grid, out_dir, n_runs=1, tag="grid"):
    
    out_dir = Path(out_dir) / tag
    rows = []

    for spec in grid:
        cells = {}
        for year in info["years"]:
            p = out_dir / f"{source}_{year}_{spec[0]}_{spec[1]}.pkl"
            with open(p, "rb") as f:
                cells[year] = pickle.load(f)

        for r in range(n_runs):
            prev = None
            for year in info["years"]:
                cell = cells[year]
                if cell is None:
                    prev = None
                    continue
                run = cell["runs"][r]
                cur = set(run["core"].tolist())

                row = {"min_share": spec[0], "max_share": spec[1],
                       "year": int(year), "run": r,
                       "V_t": cell["V_t"], "E_t": cell["E_t"],
                       "n_core": len(cur),
                       "R_t": len(cur) / cell["V_t"],
                       "S_t": run["S"], "H_t": run["H"], "Q_cp": run["Q_cp"]}
                if prev is not None and cur:
                    row["C_t"] = 1 - len(cur & prev) / len(cur)
                rows.append(row)
                prev = cur

    return pd.DataFrame(rows)

In [ ]:
GRID = list(itertools.product(
    [0.0, 0.00001, 0.00025, 0.0005, 0.001],    # min share (Kedrick et al.)
    [0.005, 0.01, 0.05, 0.1, 1.0],             # max share
))

# How many nodes/edges each grid cell keeps, on the densest year.
W_2023 = load_npz(NETWORKS_DIR / "paper" / "adj_2023.npz")
for spec in GRID:
    keep = node_mask(info["paper"], *spec)
    Wc, _ = cp_network(W_2023, keep)
    print(spec, "nodes", Wc.shape[0], "edges", Wc.nnz // 2)

In [ ]:
grid_df = {}
for source in ["news", "paper"]:
    run_grid(source, info[source], GRID, NETWORKS_DIR, CP_DIR, n_runs=1, tag="grid")
    d = compute_indicators(source, info[source], GRID, CP_DIR, n_runs=1, tag="grid")
    d["density"] = 2 * d.E_t / (d.V_t * (d.V_t - 1)).replace(0, np.nan)
    d.to_csv(CP_DIR / f"{source}_grid.csv", index=False)
    grid_df[source] = d

In [ ]:
for source, d in grid_df.items():
    print(f"\n=== {source} ===")
    _, ranking = select_spec(d)
    print("Q_cp ranking:"); print(ranking.round(4))
    print("\nnodes:"); print(d.pivot_table(index=["min_share","max_share"],
                                           columns="year", values="V_t").iloc[:, ::8])
    print("\ndensity:"); print(d.pivot_table(index=["min_share","max_share"],
                                             columns="year", values="density").iloc[:, ::8].round(3))

In [ ]:
BEST = {"news":  (0.0, 1.0),     
        "paper": (0.0, 1.0)}

main_df, band_df = {}, {}
for source, spec in BEST.items():
    run_grid(source, info[source], [spec], NETWORKS_DIR, CP_DIR, n_runs=10, tag="main")
    d = compute_indicators(source, info[source], [spec], CP_DIR, n_runs=10, tag="main")
    d.to_csv(CP_DIR / f"{source}_main.csv", index=False)
    b = d.groupby("year")[["C_t","R_t","S_t","H_t"]].agg(["mean","std","count"])
    b.to_csv(CP_DIR / f"{source}_band.csv")
    main_df[source], band_df[source] = d, b
    print(source, spec); print(b.xs("mean", axis=1, level=1).round(3).to_string())

### Visualize

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

YEAR_START, YEAR_END = 1990, 2023
YEAR_TICKS = [1990, 1995, 2000, 2005, 2010, 2015, 2020]
PERIOD_BANDS = []          # e.g. [(1998, 2006), (2006, 2017)]
EVENTS = {}                # e.g. {2012: "AlexNet", 2016: "AlphaGo"}

PANELS = [  # key, color, ylabel, math label, percent axis
    ("cit",        "#FF1F5B", "Churn of Core Nodes",      "C_{it}", False),
    ("rit",        "#00CD6C", "Relat. No. of Core Nodes", "R_{it}", True),
    ("sit",        "#009ADE", "No. of Cores",             "S_{it}", False),
    ("herfindahl", "#AF58BA", "Concentration",            "H_{it}", False),
]
METRIC_MAP = {"C_t": "cit", "R_t": "rit", "S_t": "sit", "H_t": "herfindahl"}


def to_plot_frame(df_main, z=1.96):
    """
    Collapse the (year, run) table into the mean/ci columns the plots expect.

    The band is the spread across independent detection runs. Kedrick et al.
    take theirs from cross-subfield variation, which has no counterpart here:
    with two arenas there is no field axis, so the shading reports the
    stability of the detection algorithm rather than sampling uncertainty.
    """
    g = df_main.groupby("year")[list(METRIC_MAP)].agg(["mean", "std", "count"])
    out = pd.DataFrame({"year": g.index})
    for src, dst in METRIC_MAP.items():
        out[f"{dst}_mean"] = g[(src, "mean")].values
        out[f"{dst}_ci"] = (z * g[(src, "std")] / np.sqrt(g[(src, "count")])).values
    return out.reset_index(drop=True)


def add_period_bands(ax, bands=None, color="#000000", alpha=0.05):
    for lo, hi in (bands if bands is not None else PERIOD_BANDS):
        ax.axvspan(lo, hi, color=color, alpha=alpha, lw=0, zorder=0)


def add_events(ax, events=None, fontsize=8, color="gray"):
    ev = events if events is not None else EVENTS
    top = ax.get_ylim()[1]
    for yr, label in ev.items():
        if not (YEAR_START <= yr <= YEAR_END):
            continue
        ax.axvline(yr, color=color, lw=0.8, ls="--", alpha=0.6, zorder=1)
        ax.text(yr, top, f" {label}", rotation=90, va="top", ha="left",
                fontsize=fontsize, color=color, alpha=0.9)


def plot_metric(ax, df, measure, ci_col, color, ylabel, ylim,
                title_label, pct=False):
    add_period_bands(ax)

    plot_df = df.dropna(subset=[measure])
    sns.lineplot(data=plot_df, x="year", y=measure, color=color,
                 errorbar=None, ax=ax)
    ax.fill_between(plot_df["year"],
                    plot_df[measure] - plot_df[ci_col].fillna(0),
                    plot_df[measure] + plot_df[ci_col].fillna(0),
                    color=color, alpha=0.2)

    ax.set_xlim(YEAR_START, YEAR_END)
    ax.set_xticks(YEAR_TICKS)
    ax.set_ylim(*ylim)
    ax.set_aspect(1 / ax.get_data_ratio())
    ax.set_xlabel("Year", fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.tick_params(labelsize=10)

    add_events(ax)

    if pct:
        ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1.0))
    ax.set_title(rf"${{({title_label})}}$", loc="left")


def plot_cp_indices(paper_df, news_df, ylims=None, save=None):
    """
    Figure 3.3 — structural indices over time in both arenas.

    Limits default to the data range so the panels fit whatever the current
    spec produces; pass ylims once the values have settled.
    """
    fig, axs = plt.subplots(2, 4, figsize=(14, 7))
    for row, (df, dom) in enumerate([(paper_df, "paper"), (news_df, "news")]):
        for col, (key, color, ylabel, _, pct) in enumerate(PANELS):
            m, ci = f"{key}_mean", f"{key}_ci"
            if ylims:
                ylim = ylims[dom][key]
            else:
                lo = (df[m] - df[ci].fillna(0)).min()
                hi = (df[m] + df[ci].fillna(0)).max()
                pad = 0.08 * (hi - lo) if hi > lo else 0.1
                ylim = (lo - pad, hi + pad)
            plot_metric(axs[row, col], df, m, ci, color, ylabel, ylim,
                        "abcdefgh"[row * 4 + col], pct=pct)

    fig.text(0.005, 0.975, "Paper / scientific domain", fontsize=15,
             transform=fig.transFigure)
    fig.text(0.005, 0.49, "News / social domain", fontsize=15,
             transform=fig.transFigure)
    plt.tight_layout(rect=(0.005, 0, 1, 0.98), h_pad=3, w_pad=2)

    if save:
        plt.savefig(PLOT_DIR / save, bbox_inches="tight", dpi=300)
    plt.show()

In [ ]:
main_df = {s: pd.read_csv(CP_DIR / f"{s}_main.csv") for s in ["news", "paper"]}

plot_cp_indices(to_plot_frame(main_df["paper"]),
                to_plot_frame(main_df["news"]),
                save="fig_cp_indices.png")

In [ ]:
# YLIMS = {
#     "paper": {"cit": (0.4, 0.7), "rit": (0.35, 0.45),
#               "sit": (1000, 2300), "herfindahl": (0.0, 0.23)},
#     "news":  {"cit": (0.4, 0.7), "rit": (0.5, 0.7),
#               "sit": (0, 1300), "herfindahl": (0.0, 0.23)},
# }
# plot_cp_indices(to_plot_frame(main_df["paper"]), to_plot_frame(main_df["news"]), ylims=YLIMS, save="fig_cp_indices.png")